# 03 — KnottedGraph vs Topoly: focused paper scaling

This notebook performs **one benchmark experiment** and reuses its results for three paper views:

1. **Crossing scaling:** runtime versus projected crossing count $c$, aggregated across a fixed panel of heterogeneous connected trivalent graphs.
2. **Vertex scaling:** runtime versus $V$ at one selected fixed crossing count.
3. **Edge scaling:** the exact same fixed-$c$ rows replotted versus $E$.

No separate $K_4$, prism, edge-theta, throughput, or random-cubic benchmark is executed. This avoids redundant Yamada calculations.

At each projected crossing count, the same panel of connected trivalent graphs is used. Graph construction and graph-to-PD conversion happen **outside** the timed region, and KnottedGraph and Topoly receive the same PD code.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, os, subprocess, sys

from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
DEV = ROOT / "dev"
for path in (SRC, DEV):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import knotted_graph

kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install Topoly first: pip install topoly") from exc

OUT = ROOT / "User_guide" / "benchmarks"
RES = OUT / "results_latest"
FIG = OUT / "figures_latest"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("KnottedGraph:", kg_path)
print("Topoly:", Path(topoly.__file__).resolve())


## 1. Configuration

The key controls are now directly selectable here.

- `MAX_PROJECTED_CROSSINGS` controls the largest projected crossing count attempted by the crossing-scaling benchmark.
- `SIZE_SCALING_CROSSINGS` selects which already-computed crossing slice is reused for the vertex- and edge-scaling plots.
- `CROSSING_GRAPHS` controls how many different connected trivalent graph sizes are evaluated at each crossing count.

For a normal local paper run the default is 21 graph sizes per $c$. GitHub Actions automatically uses a smaller smoke configuration.

During execution, a persistent progress bar remains visible while each completed sample also prints KnottedGraph time, Topoly time, and speedup.


In [ ]:
IS_CI = os.environ.get("CI", "").lower() == "true"

PROFILE = "smoke" if IS_CI else "paper"
CROSSING_GRAPHS = 3 if IS_CI else 21

# Change these two values directly in the notebook.
MAX_PROJECTED_CROSSINGS = 12 if IS_CI else 80
SIZE_SCALING_CROSSINGS = 8

TIMEOUT_S = 10 if IS_CI else 120
CENSOR_FRONTIER = 2

raw_csv = RES / "topoly_yamada_paper_scaling_raw.csv"
aggregate_csv = RES / "topoly_yamada_paper_scaling_aggregate.csv"

if SIZE_SCALING_CROSSINGS > MAX_PROJECTED_CROSSINGS:
    raise ValueError("SIZE_SCALING_CROSSINGS must be <= MAX_PROJECTED_CROSSINGS")

print(
    f"mode={PROFILE}, graph sizes/c={CROSSING_GRAPHS}, "
    f"max crossings={MAX_PROJECTED_CROSSINGS}, "
    f"size-scaling slice c={SIZE_SCALING_CROSSINGS}, "
    f"timeout/framework/sample={TIMEOUT_S}s"
)


In [ ]:
def _load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def run_streamed_benchmark(cmd, *, total, description):
    print("Running:", " ".join(map(str, cmd)))
    process = subprocess.Popen(
        cmd, cwd=ROOT, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Benchmark subprocess did not expose stdout.")

    streamed_rows = []
    summary_rows = None
    bar = tqdm(
        total=total, desc=description, unit="sample",
        dynamic_ncols=True, leave=True, position=0,
    )

    def _time_text(row, framework):
        status = row.get(f"{framework}_status", "?")
        value = row.get(f"{framework}_s")
        if status == "ok" and value is not None:
            return f"{float(value):.6f} s"
        if status == "timeout":
            return f"TIMEOUT (>={row.get('timeout_s', '?')} s)"
        if status == "error":
            return "ERROR"
        if status == "skipped_after_censor_frontier":
            return "SKIPPED"
        return status.upper()

    for raw_line in process.stdout:
        line = raw_line.rstrip()
        if not line:
            continue
        if line.startswith("SUMMARY="):
            summary_rows = json.loads(line[len("SUMMARY="):])
            continue
        if line.startswith("{"):
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                bar.write(line)
                continue
            if "family" in row:
                streamed_rows.append(row)
                sample_id = row.get("sample", "?")
                meta = " ".join(
                    f"{key}={row[key]}"
                    for key in ("V", "E", "crossings")
                    if row.get(key) is not None
                )

                kg_text = _time_text(row, "knottedgraph")
                tp_text = _time_text(row, "topoly")
                if (
                    row.get("knottedgraph_status") == "ok"
                    and row.get("topoly_status") == "ok"
                    and row.get("knottedgraph_s")
                    and row.get("topoly_s")
                ):
                    speedup = float(row["topoly_s"]) / float(row["knottedgraph_s"])
                    speedup_text = f" | Topoly/KnottedGraph={speedup:.2f}x"
                else:
                    speedup_text = ""

                bar.write(
                    f"TIMING {row.get('family')} {meta} sample={sample_id} | "
                    f"KnottedGraph={kg_text} | Topoly={tp_text}{speedup_text}"
                )
                bar.set_postfix_str(
                    f"c={row.get('crossings')} V={row.get('V')} E={row.get('E')} "
                    f"KG={kg_text}, T={tp_text}",
                    refresh=False,
                )
                bar.update(1)
                bar.refresh()
                continue
        bar.write(line)

    return_code = process.wait()
    if bar.n < bar.total:
        bar.set_postfix_str("completed with censored/skipped frontier", refresh=False)
    bar.refresh()
    bar.close()

    if return_code:
        raise RuntimeError(
            f"Benchmark failed with exit code {return_code}: {' '.join(map(str, cmd))}"
        )

    rows = summary_rows if summary_rows is not None else streamed_rows
    if not rows:
        raise RuntimeError("Benchmark completed without sample rows.")
    print(f"completed {len(rows)} sample records")
    return rows


## 2. Run the single reusable benchmark ensemble

At every crossing count $c$, the benchmark uses the same graph-size panel. The local paper default uses 21 connected trivalent graphs with different $V$ and $E$. Because every graph is trivalent,

$$
E=\frac{3V}{2}.
$$

The same $(V,E)$ panel is reused at every $c$. This lets us examine crossing complexity while averaging over graph-size heterogeneity, and then reuse one fixed-$c$ slice for size scaling without performing a second benchmark.


In [ ]:
paper_script = ROOT / "dev" / "benchmark_topoly_paper_scaling.py"
env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join([str(SRC), str(DEV)])
env["PYTHONNOUSERSITE"] = "1"

paper_module = _load_module(
    paper_script,
    "kg_topoly_paper_scaling_notebook_plan",
)
plan = paper_module.paper_plan(
    PROFILE,
    CROSSING_GRAPHS,
    MAX_PROJECTED_CROSSINGS,
    SIZE_SCALING_CROSSINGS,
)

assert set(plan) == {"crossings_graph_ensemble"}
assert max(plan["crossings_graph_ensemble"]["x_values"]) == MAX_PROJECTED_CROSSINGS
assert SIZE_SCALING_CROSSINGS in plan["crossings_graph_ensemble"]["x_values"]

total = (
    len(plan["crossings_graph_ensemble"]["x_values"])
    * CROSSING_GRAPHS
)

print("Crossing grid:", plan["crossings_graph_ensemble"]["x_values"])
print("Total graph samples to attempt:", total)

cmd = [
    sys.executable, str(paper_script),
    "--profile", PROFILE,
    "--crossing-graphs", str(CROSSING_GRAPHS),
    "--max-crossings", str(MAX_PROJECTED_CROSSINGS),
    "--size-scaling-crossings", str(SIZE_SCALING_CROSSINGS),
    "--timeout", str(TIMEOUT_S),
    "--censor-frontier", str(CENSOR_FRONTIER),
]

rows = run_streamed_benchmark(
    cmd, total=total, description="Paper Yamada scaling"
)


## 3. Acceptance checks and raw-data export

The notebook verifies that only `crossings_graph_ensemble` was evaluated, that the same heterogeneous $(V,E)$ panel is reused across crossing counts, and that the selected size-scaling slice is present.


In [ ]:
from collections import defaultdict

assert {row["family"] for row in rows} == {"crossings_graph_ensemble"}

groups = defaultdict(list)
for row in rows:
    groups[int(row["crossings"])].append(row)

expected_panel = None
for crossings, group in sorted(groups.items()):
    assert len(group) == CROSSING_GRAPHS, (crossings, len(group), CROSSING_GRAPHS)
    assert len({int(row["sample"]) for row in group}) == CROSSING_GRAPHS
    panel = sorted((int(row["V"]), int(row["E"])) for row in group)
    assert len(set(panel)) == CROSSING_GRAPHS
    assert all(int(row["crossings"]) == crossings for row in group)
    assert all(bool(row["connected"]) for row in group)
    assert all(int(row["regular_degree"]) == 3 for row in group)
    assert all(int(row["E"]) == 3 * int(row["V"]) // 2 for row in group)
    if expected_panel is None:
        expected_panel = panel
    else:
        assert panel == expected_panel, (crossings, panel, expected_panel)

assert SIZE_SCALING_CROSSINGS in groups, (
    SIZE_SCALING_CROSSINGS, sorted(groups)
)

for row in rows:
    if row["correctness"] == "PASS":
        assert row["knottedgraph_status"] == "ok"
        assert row["topoly_status"] == "ok"
        assert row["pd_hash"]

keys = list(dict.fromkeys(key for row in rows for key in row))
with raw_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys)
    writer.writeheader()
    writer.writerows(rows)

print("PASS: only crossings_graph_ensemble was evaluated.")
print("PASS: the same heterogeneous V/E panel is reused across crossing counts.")
print("V/E panel:", expected_panel)
print(f"PASS: c={SIZE_SCALING_CROSSINGS} is available for V/E scaling.")
print(f"wrote {len(rows)} sample-level records to {raw_csv}")


## 4. Generate three plots from the same calculations

The plotting stage produces:

1. `topoly_vs_knottedgraph_crossings.{png,pdf}` — median runtime versus projected crossings across the graph-size panel.
2. `topoly_vs_knottedgraph_vertices.{png,pdf}` — runtime versus $V$ using only the selected fixed-$c$ slice.
3. `topoly_vs_knottedgraph_edges.{png,pdf}` — the same fixed-$c$ rows replotted versus $E$.

The vertex and edge figures are therefore **not independent experiments**: for trivalent graphs $E=3V/2$. They are two parameterizations of the same size-scaling data.


In [ ]:
plot_script = ROOT / "dev" / "plot_topoly_paper_scaling.py"
plot_cmd = [
    sys.executable, str(plot_script), str(raw_csv),
    "--figure-dir", str(FIG),
    "--aggregate-csv", str(aggregate_csv),
    "--size-crossings", str(SIZE_SCALING_CROSSINGS),
]

print("Running:", " ".join(plot_cmd))
plot_process = subprocess.Popen(
    plot_cmd, cwd=ROOT, env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
if plot_process.stdout is None:
    raise RuntimeError("Paper-quality plot subprocess did not expose stdout.")
for line in plot_process.stdout:
    print(line, end="")
if plot_process.wait():
    raise RuntimeError("Paper-quality plot generation failed.")

expected_stems = [
    "topoly_vs_knottedgraph_crossings",
    "topoly_vs_knottedgraph_vertices",
    "topoly_vs_knottedgraph_edges",
]
for stem in expected_stems:
    assert (FIG / f"{stem}.png").exists(), stem
    assert (FIG / f"{stem}.pdf").exists(), stem
assert aggregate_csv.exists()
print("PASS: crossing, vertex, and edge figure PNG/PDF pairs were created.")


## 5. Interpretation

**Crossing view.** At each $c$, runtime is aggregated across the same heterogeneous connected-trivalent graph-size panel. This reduces dependence on a single abstract graph while holding the size distribution fixed across $c$.

**Vertex and edge views.** These use the exact same rows at $c=$ `SIZE_SCALING_CROSSINGS`, so crossing complexity is fixed while graph size changes. Because the benchmark graphs are trivalent, $E=3V/2$; consequently the $V$ and $E$ plots are equivalent reparameterizations and should not be presented as independent evidence.

Only Yamada evaluation is timed. Graph construction and the common PD-code construction are excluded.
